# Cloud-Optimized Data Preparation Workshop

## Overview
This notebook walks through converting traditional geospatial data formats into cloud-optimized alternatives that enable efficient web-based visualization via HTTP range requests.

### Data Sources Used in This Workshop

This workshop uses publicly available data:

1. **Santa Clara County Road Network (Vector)** - [RoadCenterLine Dataset](https://data.sccgov.org/Transportation/RoadCenterLine/amyq-ahrs)
   - Public transportation infrastructure data
   - Open Data portal (Socrata)
   - Available in multiple formats including shapefile

2. **Stanford Digital Repository Reference** - [World PMTiles](https://purl.stanford.edu/hf224mw4004)
   - Large-scale example of cloud-optimized data (108 GiB)
   - Demonstrates the full potential of PMTiles format
   - Live URL: `https://stacks.stanford.edu/file/druid:hf224mw4004/20231116.pmtiles`

3. **David Rumsey Historical Maps (Optional Raster)** - [Santa Clara County Maps](https://www.davidrumsey.com/luna/servlet/detail/RUMSEY~8~1~1578~170036:San-Mateo,-Santa-Cruz,-Santa-Clara)
   - Georeferenced historical maps for comparison with contemporary data
   - Example of raster data suitable for COG conversion
   - Demonstrates temporal analysis capabilities

### What We'll Do
1. **Examine source data** using GDAL command-line tools
2. **Convert vector data** (shapefiles) → GeoJSON → PMTiles
3. **Convert raster data** (`santa_clara_1896.tif`) → Cloud-Optimized GeoTIFF (COG)
4. **Generate metadata** and prepare URLs for web deployment

### Key Formats
- **PMTiles**: Vector tile container with embedded index (for efficient HTTP range requests)
- **GeoJSON**: Easy-to-read exchange format that Tippecanoe can tile directly

### Output
- `santa_clara_roads.pmtiles` - Vector tile dataset of road network
- `santa_clara_1896_cog.tif` - Cloud-optimized GeoTIFF derived from the historical map
- HTTP URLs ready for web map deployment


## Section 1: Install Dependencies and Import Libraries


In [ ]:
# Install required packages
# Run this cell first in Google Colab.

# This notebook is designed for Colab. The setup below installs every
# command-line tool and Python library that the later cells depend on, so
# students do not need to install anything on their own computers.

import multiprocessing
import os
import shutil
import subprocess
import sys
from pathlib import Path

RUN_IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ

if not RUN_IN_COLAB:
    raise RuntimeError(
        "This workshop notebook is intended to run in Google Colab. "
        "Open it in Colab so the setup cell can install GDAL and Tippecanoe "
        "inside the /content environment."
    )

print("Detected Google Colab. Installing the notebook's single setup path...")

# GDAL/OGR provides command-line tools such as ogrinfo, ogr2ogr, and
# gdal_translate. We use these to inspect, reproject, and convert geospatial
# files without asking students to install desktop GIS software.
print("Installing GDAL/OGR and build tools...")
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run([
    'apt-get', 'install', '-y', '-qq',
    'build-essential',
    'git',
    'gdal-bin',
    'libgdal-dev',
    'libsqlite3-dev',
    'zlib1g-dev',
], check=True)

# Tippecanoe creates vector tiles from GeoJSON. For PMTiles output support,
# this notebook builds the actively maintained Felt version from source.
print("Building Tippecanoe from https://github.com/felt/tippecanoe ...")
tippecanoe_source = Path('/content/tippecanoe')
if tippecanoe_source.exists():
    shutil.rmtree(tippecanoe_source)

subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/felt/tippecanoe.git',
    str(tippecanoe_source),
], check=True)

# Use all available Colab CPU cores to keep the source build reasonably quick.
subprocess.run(
    ['make', f'-j{multiprocessing.cpu_count()}'],
    cwd=tippecanoe_source,
    check=True,
)
subprocess.run(['make', 'install'], cwd=tippecanoe_source, check=True)

tippecanoe_path = shutil.which('tippecanoe')
if not tippecanoe_path:
    raise RuntimeError(
        "Tippecanoe finished building, but the tippecanoe executable was not "
        "found on PATH. Check the build output above before continuing."
    )

version_result = subprocess.run(
    [tippecanoe_path, '--version'],
    capture_output=True,
    text=True,
)
version_message = (version_result.stdout or version_result.stderr).strip().splitlines()
print(f"Tippecanoe executable: {tippecanoe_path}")
if version_message:
    print(version_message[0])

# The packages below are Python libraries. They let us load spatial data into
# data frames, inspect coordinate systems, and work with vector/raster files
# directly from Python.
packages = [
    'geopandas',      # Vector data manipulation
    'rasterio',       # Raster data handling
    'shapely',        # Geometric operations
    'pandas',         # Data analysis tables
    'pyproj',         # Coordinate reference systems and transformations
]

print("Installing required Python packages...")
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("Setup complete: GDAL/OGR, Felt Tippecanoe, and Python packages are ready.")


In [70]:
# Colab setup: copy the workshop data into /content/data
# This cell only needs to run in Google Colab.

import os
import shutil
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
GITHUB_REPO = 'https://github.com/mapninja/geo4lib_cloud_optimized.git'

# Colab sets this environment variable, so we can keep the notebook
# lightweight when it is run locally.
RUN_IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ

if RUN_IN_COLAB:
    # Remove any old copy so the notebook always starts from fresh data.
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)

    repo_dir = PROJECT_ROOT / '_geo4lib_cloud_optimized_repo'
    if repo_dir.exists():
        shutil.rmtree(repo_dir)

    print("Downloading the workshop repository from GitHub...")
    subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, str(repo_dir)], check=True)

    # Copy only the shared data folder into /content/data so the rest of
    # the notebook can use the same paths as the local repository.
    shutil.copytree(repo_dir / 'data', DATA_DIR)
    shutil.rmtree(repo_dir)

    print(f"✓ Data copied to {DATA_DIR}")
else:
    print("Running outside Colab. The notebook will use the local data/ folder in this repository.")
    print(f"Data directory: {DATA_DIR}")


✓ Data copied to /content/data


In [71]:
# Import required libraries

import os
import subprocess
import json
from pathlib import Path
from typing import Dict, Tuple

# Geospatial libraries
import geopandas as gpd
import rasterio
from rasterio.io import MemoryFile
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

# Data manipulation
import pandas as pd
from shapely.geometry import box

# System utilities
import logging
import warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Define project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
SCRATCH_DIR = PROJECT_ROOT / 'scratch'

# Ensure directories exist
DATA_DIR.mkdir(exist_ok=True)
SCRATCH_DIR.mkdir(exist_ok=True)

print("✓ All libraries imported successfully!")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

✓ All libraries imported successfully!
Project root: /content
Data directory: /content/data


## Section 2: Examine and Load Source Data

### Step 1: Inspect Shapefile with GDAL

First, we'll use GDAL command-line tools to examine the shapefile structure. This shows us:
- Layer names and feature counts
- Geometry types
- Coordinate reference system (CRS)
- Attribute fields and their data types
- Spatial extent (bounding box)


In [81]:
# Examine the shapefile using GDAL command-line tools
# This step inspects the source data without loading it into memory

shapefile_path = DATA_DIR / 'RoadCenterLine_20260610' / 'geo_export_71410037-f1cd-48fb-a6ed-f63e86f313c2.shp'

print(f"Examining shapefile: {shapefile_path}")
print("=" * 70)

# Use ogrinfo to get detailed information
result = subprocess.run(
    ['ogrinfo', str(shapefile_path)],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
else:
    print("Error running ogrinfo. Make sure GDAL is installed:")
    print(result.stderr)
    print("\nInstallation instructions:")
    print("  macOS: brew install gdal")
    print("  Ubuntu: sudo apt-get install gdal-bin")

Examining shapefile: /content/data/RoadCenterLine_20260610/geo_export_71410037-f1cd-48fb-a6ed-f63e86f313c2.shp
INFO: Open of `/content/data/RoadCenterLine_20260610/geo_export_71410037-f1cd-48fb-a6ed-f63e86f313c2.shp'
      using driver `ESRI Shapefile' successful.
1: geo_export_71410037-f1cd-48fb-a6ed-f63e86f313c2 (Line String)



In [82]:
# Load the shapefile using GeoPandas
# GeoPandas wraps GDAL/OGR for easier Python data manipulation

logger.info("Loading shapefile into GeoPandas...")
gdf = gpd.read_file(shapefile_path)

print(f"\nDataset Summary:")
print(f"  • Feature count: {len(gdf)}")
print(f"  • Geometry type: {gdf.geometry.type.unique()}")
print(f"  • Coordinate Reference System (CRS): {gdf.crs}")
print(f"  • Spatial extent (bounds):")
bounds = gdf.total_bounds
print(f"    - West: {bounds[0]:.6f}, South: {bounds[1]:.6f}")
print(f"    - East: {bounds[2]:.6f}, North: {bounds[3]:.6f}")

print(f"\nColumn names and types:")
print(gdf.dtypes)

print(f"\nFirst few features:")
print(gdf.head())


Dataset Summary:
  • Feature count: 72531
  • Geometry type: ['LineString']
  • Coordinate Reference System (CRS): EPSG:4326
  • Spatial extent (bounds):
    - West: -122.202515, South: 36.913564
    - East: -121.211531, North: 37.484712

Column names and types:
objectid       float64
roadclass       object
fullrdname      object
geometry      geometry
dtype: object

First few features:
   objectid   roadclass       fullrdname  \
0   13942.0  Local_road    LINDENTREE LN   
1   23094.0  Local_road  GLEN EXETER WAY   
2   46433.0  Local_road    LINDA MESA DR   
3   22869.0  Local_road  REGENCY OAKS DR   
4   22472.0  Local_road       HAMPTON DR   

                                            geometry  
0  LINESTRING (-121.96821 37.33243, -121.96936 37...  
1  LINESTRING (-121.80716 37.3318, -121.8072 37.3...  
2  LINESTRING (-121.61867 37.13781, -121.61893 37...  
3  LINESTRING (-122.01213 37.30703, -122.01214 37...  
4  LINESTRING (-122.01316 37.34828, -122.01316 37...  


## Section 3: Data Validation and Coordinate System Transformation

### Why Transform Coordinates?

Web maps require data in Web Mercator (EPSG:3857) or similar web-friendly projections.
Our data likely uses a local projected coordinate system (like UTM or State Plane).
We need to transform it to a web-compatible system.

**Key points:**
- Original CRS: Likely UTM or local projection
- Target CRS: EPSG:3857 (Web Mercator, used by most web mapping libraries)
- This transformation preserves accuracy while enabling web compatibility


In [74]:
# Validate the data

logger.info("Validating data integrity...")

# Check for null geometries
null_geoms = gdf.geometry.isnull().sum()
print(f"\nData Validation:")
print(f"  • Null geometries: {null_geoms}")

# Check for invalid geometries
invalid_geoms = (~gdf.geometry.is_valid).sum()
print(f"  • Invalid geometries: {invalid_geoms}")

# Check for empty geometries
empty_geoms = (gdf.geometry.is_empty).sum()
print(f"  • Empty geometries: {empty_geoms}")

# Display current CRS
print(f"\nCurrent Coordinate Reference System:")
print(f"  • EPSG Code: {gdf.crs.to_epsg()}")
print(f"  • CRS Name: {gdf.crs.to_string()}")

# Check if already in Web Mercator
if gdf.crs.to_epsg() == 3857:
    print("\n✓ Data is already in Web Mercator (EPSG:3857)")
    gdf_web = gdf.copy()
else:
    print(f"\n→ Transforming from EPSG:{gdf.crs.to_epsg()} to Web Mercator (EPSG:3857)...")
    gdf_web = gdf.to_crs(epsg=3857)
    print("✓ Transformation complete")
    print(f"  New extent: {gdf_web.total_bounds}")


Data Validation:
  • Null geometries: 0
  • Invalid geometries: 0
  • Empty geometries: 0

Current Coordinate Reference System:
  • EPSG Code: 4326
  • CRS Name: EPSG:4326

→ Transforming from EPSG:4326 to Web Mercator (EPSG:3857)...
✓ Transformation complete
  New extent: [-13603521.78221348   4427065.6090618  -13493205.89966292
   4506886.50848732]


## Section 4: Convert Vector Data to PMTiles

### The Conversion Pipeline

```
Shapefile → GeoJSON → PMTiles
  ↓          ↓           ↓
OGR Format  Geographic  Cloud-ready
             Features    Vector Tiles
```

### Process Steps

1. **Export to GeoJSON**: Convert shapefile to a widely supported exchange format
   - GeoJSON is easy for Tippecanoe to read
   - We also transform to EPSG:4326 so the coordinates are geographic

2. **Create PMTiles**: Generate the final cloud-optimized format
   - PMTiles bundles all tiles in a single file
   - Includes an internal directory at the start of the file
   - Enables HTTP range requests to fetch only needed tiles


In [75]:
# Step 1: Export the shapefile to GeoJSON
# Tippecanoe reads GeoJSON directly, so this is the most compatible input.

logger.info("Converting shapefile to GeoJSON...")

# Save our web-mercator data to a temporary shapefile first.
# We then convert it to WGS84 GeoJSON for Tippecanoe.
temp_shp = SCRATCH_DIR / 'temp_roads_3857.shp'
gdf_web.to_file(temp_shp)

# Path for the GeoJSON intermediate format
geojson_path = SCRATCH_DIR / 'roads.geojson'

# Use GDAL's ogr2ogr to convert to GeoJSON in WGS84 (EPSG:4326).
cmd_geojson = [
    'ogr2ogr',
    '-t_srs', 'EPSG:4326',
    '-f', 'GeoJSON',
    str(geojson_path),
    str(temp_shp),
    '-nln', 'roads'
]

print(f"Running: {' '.join(cmd_geojson)}")
result = subprocess.run(cmd_geojson, capture_output=True, text=True)

if result.returncode == 0:
    print(f"✓ GeoJSON created: {geojson_path}")
    print(f"  File size: {geojson_path.stat().st_size / 1024:.2f} KB")
else:
    print("Error:", result.stderr)
    print("\nNote: If ogr2ogr is not found, install GDAL:")
    print("  macOS: brew install gdal")
    print("  Ubuntu: sudo apt-get install gdal-bin")


Running: ogr2ogr -t_srs EPSG:4326 -f GeoJSON /content/scratch/roads.geojson /content/scratch/temp_roads_3857.shp -nln roads
✓ GeoJSON created: /content/scratch/roads.geojson
  File size: 80459.42 KB


In [ ]:
# Step 2: Create PMTiles from GeoJSON
# We'll use the command-line approach because Tippecanoe is the standard
# tool for turning GeoJSON features into web map vector tiles.

logger.info("Creating PMTiles from GeoJSON...")

import shutil

pmtiles_path = DATA_DIR / 'santa_clara_roads.pmtiles'
tippecanoe_executable = shutil.which('tippecanoe')

if not tippecanoe_executable:
    raise RuntimeError(
        "Tippecanoe is not available. Re-run the first setup cell so Colab "
        "can build and install Tippecanoe from https://github.com/felt/tippecanoe."
    )

if not geojson_path.exists():
    raise FileNotFoundError(
        f"Expected GeoJSON input was not found: {geojson_path}. "
        "Run the previous GeoJSON export cell before creating PMTiles."
    )

cmd_pmtiles = [
    tippecanoe_executable,
    '-zg',                        # Let Tippecanoe choose a good max zoom
    '--projection=EPSG:4326',     # Tell Tippecanoe the input is geographic
    '-f',                         # Overwrite any old PMTiles file from the repo
    '-o', str(pmtiles_path),      # Output PMTiles file
    '-l', 'roads',                # Layer name used by MapLibre
    '--drop-densest-as-needed',   # Reduce dense tiles if needed for web use
    str(geojson_path)             # Input GeoJSON file
]

print(f"Command: {' '.join(cmd_pmtiles)}")
result = subprocess.run(cmd_pmtiles, capture_output=True, text=True)

if result.returncode == 0:
    output_size = pmtiles_path.stat().st_size
    print(f"\nPMTiles created successfully: {pmtiles_path}")
    print(f"  File size: {output_size / (1024*1024):.2f} MB")
    print("  Layer: roads")

    # A tiny PMTiles file usually means Tippecanoe did not actually read
    # any features. Stop here so students do not publish a broken tile file.
    if output_size < 100_000:
        raise RuntimeError(
            "The PMTiles file is unexpectedly small. Check that the GeoJSON "
            "contains road features and that Tippecanoe did not report an error."
        )
else:
    print("Tippecanoe error output:")
    print(result.stderr)
    raise RuntimeError("Tippecanoe failed. Read the error output above before continuing.")


In [84]:
# Verify PMTiles file exists and check its properties

logger.info("Verifying PMTiles output...")

if pmtiles_path.exists():
    file_size = pmtiles_path.stat().st_size
    size_mb = file_size / (1024 * 1024)
    size_gb = file_size / (1024 * 1024 * 1024)

    print(f"✓ PMTiles file created successfully!")
    print(f"\n  Path: {pmtiles_path}")
    print(f"  File size: {size_mb:.2f} MB" if size_mb < 1024 else f"  File size: {size_gb:.2f} GB")
    print(f"\n  Metadata:")
    print(f"  • Layer name: roads")
    print(f"  • Source: {shapefile_path.name}")
    print(f"  • Features: {len(gdf_web):,}")
    print(f"  • Geometry type: LineString (roads)")
    print(f"  • Coordinate system: Web Mercator (EPSG:3857)")
    print(f"  • Bounds: {gdf_web.total_bounds}")

    # Get more details using ogrinfo if available
    print(f"\nDetailed layer information:")
    cmd = ['ogrinfo', '-json', str(pmtiles_path)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and result.stdout:
        try:
            info = json.loads(result.stdout)
            if 'layers' in info and len(info['layers']) > 0:
                layer = info['layers'][0]
                print(f"  • Feature count: {layer.get('featureCount', 'unknown')}")
        except:
            pass
else:
    print("⚠️  PMTiles file not found at expected path:")
    print(f"   {pmtiles_path}")
    print("\nMake sure tippecanoe is installed and the previous cell ran successfully.")

✓ PMTiles file created successfully!

  Path: /content/data/santa_clara_roads.pmtiles
  File size: 10.84 MB

  Metadata:
  • Layer name: roads
  • Source: geo_export_71410037-f1cd-48fb-a6ed-f63e86f313c2.shp
  • Features: 72,531
  • Geometry type: LineString (roads)
  • Coordinate system: Web Mercator (EPSG:3857)
  • Bounds: [-13603521.78221348   4427065.6090618  -13493205.89966292
   4506886.50848732]

Detailed layer information:
  • Feature count: 85486


## Section 5: Convert Raster Data to Cloud-Optimized GeoTIFF

### Why Make a COG?

A Cloud-Optimized GeoTIFF stores raster data in a layout that web tools can read efficiently over HTTP.
That means a map viewer can request only the pieces it needs instead of downloading the whole image.

In this example, we convert `data/santa_clara_1896.tif` into a COG for faster web delivery and zoom-based access.


In [85]:
# Convert the historical raster into a Cloud-Optimized GeoTIFF.
# We use GDAL's COG output driver so the file is organized for web access.

source_tif = DATA_DIR / 'santa_clara_1896.tif'
cog_path = DATA_DIR / 'santa_clara_1896_cog.tif'

print(f"Source raster: {source_tif}")
print(f"COG output: {cog_path}")

# Remove an older output file first so reruns stay predictable for students.
if cog_path.exists():
    cog_path.unlink()

cmd_cog = [
    'gdal_translate',
    '-of', 'COG',                # Tell GDAL to create a Cloud-Optimized GeoTIFF
    '-co', 'COMPRESS=LZW',       # Compress pixels to reduce file size
    '-co', 'RESAMPLING=AVERAGE', # Build overviews using averaging for smoother zoomed-out views
    '-co', 'LEVEL=9',            # Allow internal pyramid levels for efficient zooming
    str(source_tif),             # Input raster
    str(cog_path)                # Output COG file
]

print(f"Running: {' '.join(cmd_cog)}")
result = subprocess.run(cmd_cog, capture_output=True, text=True)

if result.returncode == 0:
    print(f"\n✓ COG created successfully: {cog_path}")
    print(f"  File size: {cog_path.stat().st_size / (1024*1024):.2f} MB")
else:
    print("Error output:", result.stderr)


Source raster: /content/data/santa_clara_1896.tif
COG output: /content/data/santa_clara_1896_cog.tif
Running: gdal_translate -of COG -co COMPRESS=LZW -co RESAMPLING=AVERAGE -co LEVEL=9 /content/data/santa_clara_1896.tif /content/data/santa_clara_1896_cog.tif

✓ COG created successfully: /content/data/santa_clara_1896_cog.tif
  File size: 13.77 MB


In [86]:
# Verify the Cloud-Optimized GeoTIFF exists and inspect its basic properties.
# This helps students confirm that the conversion worked before they move on.

if cog_path.exists():
    file_size = cog_path.stat().st_size
    size_mb = file_size / (1024 * 1024)
    print("✓ COG file created successfully!")
    print(f"\n  Path: {cog_path}")
    print(f"  File size: {size_mb:.2f} MB")

    with rasterio.open(cog_path) as src:
        # These properties help us confirm that the raster is still georeferenced.
        print("\n  Raster metadata:")
        print(f"  • Width: {src.width}")
        print(f"  • Height: {src.height}")
        print(f"  • Band count: {src.count}")
        print(f"  • CRS: {src.crs}")
        print(f"  • Bounds: {src.bounds}")
        print(f"  • Driver: {src.driver}")
else:
    print("⚠️  COG file not found at expected path:")
    print(f"   {cog_path}")
    print("\nMake sure gdal_translate ran successfully in the previous cell.")


✓ COG file created successfully!

  Path: /content/data/santa_clara_1896_cog.tif
  File size: 13.77 MB

  Raster metadata:
  • Width: 2098
  • Height: 2365
  • Band count: 3
  • CRS: EPSG:3857
  • Bounds: BoundingBox(left=-13652383.11953007, bottom=4421730.386864315, right=-13524470.553019425, top=4565921.626043379)
  • Driver: GTiff


## Prepare Data for Download


In [87]:
import shutil

output_archive_name = 'cloud_optimized_data'
output_archive_format = 'zip'

print(f"Zipping the 'data' directory into '{output_archive_name}.{output_archive_format}'...")
archive_path = shutil.make_archive(
    base_name=output_archive_name,
    format=output_archive_format,
    root_dir=PROJECT_ROOT,
    base_dir='data'
)

print(f"✓ Data zipped to: {archive_path}")
print("You can now download this file from the file browser on the left (if in Colab) or access it directly at the path indicated.")

Zipping the 'data' directory into 'cloud_optimized_data.zip'...
✓ Data zipped to: /content/cloud_optimized_data.zip
You can now download this file from the file browser on the left (if in Colab) or access it directly at the path indicated.


## Useful Resources and References

### Cloud-Optimized Data Formats
- [PMTiles Specification](https://github.com/protomaps/PMTiles)
- [Cloud Optimized GeoTIFF](https://www.cogeo.org/)
- [GeoJSON Format](https://geojson.org/)

### Tools
- [GDAL/OGR Documentation](https://gdal.org/)
- [Tippecanoe by Felt (PMTiles creator)](https://github.com/felt/tippecanoe)
- [MapLibre GL JS](https://maplibre.org/)

### Performance Optimization
- [HTTP Range Requests MDN](https://developer.mozilla.org/en-US/docs/Web/HTTP/Headers/Range)
- [Cloud Native Geospatial](https://cloudnativegeo.org/)
- [Stacks Documentation](https://purl.stanford.edu/)

### Data Sources Used in This Workshop
- **Vector**: [Santa Clara County RoadCenterLine](https://data.sccgov.org/Transportation/RoadCenterLine/amyq-ahrs) (Socrata Open Data)
- **Reference**: [Stanford Digital Repository - World PMTiles](https://purl.stanford.edu/hf224mw4004) (108 GB scale example)
- **Historical Maps**: [David Rumsey Map Collection](https://www.davidrumsey.com/luna/servlet/detail/RUMSEY~8~1~1578~170036:San-Mateo,-Santa-Cruz,-Santa-Clara) (Optional raster overlay)

### Additional Data Sources
- [Stanford Digital Repository](https://purl.stanford.edu/)
- [OpenStreetMap Data](https://www.openstreetmap.org/)
- [USGS Data Repository](https://www.usgs.gov/products/maps/gis-data)

---

**Completed!** Your data is ready for web deployment.
Next: Update `index.html` with your GitHub URLs and enable GitHub Pages to go live.
